# moomoo User ID Batch Extractor

这个 notebook 用于在本地批量扫描 `/Users/xiao1/Desktop/moomoo id` 里的照片，提取 8 位 `User ID`，并生成 CSV。

In [ ]:
from pathlib import Path
import csv
import re
import subprocess
import tempfile
from PIL import Image, ImageFilter, ImageOps

INPUT_DIR = Path('/Users/xiao1/Desktop/moomoo id')
OUTPUT_CSV = INPUT_DIR / 'moomoo_user_ids.csv'
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff', '.heic', '.heif'}

KEYWORD_PATTERN = re.compile(r'user\s*id|moomoo\s*id|\bid\b', re.IGNORECASE)
DIRECT_PATTERNS = [
    re.compile(r'user\s*id\D{0,8}(\d{8})', re.IGNORECASE),
    re.compile(r'(\d{8})\s*\n?\s*user\s*id', re.IGNORECASE),
]
ID_PATTERNS = [
    re.compile(r'user\s*id\D{0,8}(\d{8})', re.IGNORECASE),
    re.compile(r'moomoo\s*id\D{0,8}(\d{8})', re.IGNORECASE),
    re.compile(r'\bid\D{0,6}(\d{8})\b', re.IGNORECASE),
    re.compile(r'\b(\d{8})\b'),
]


In [ ]:
def list_image_files(folder: Path):
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

def preprocess_image(image: Image.Image) -> Image.Image:
    image = image.convert('L')
    width, height = image.size
    image = image.resize((width * 2, height * 2))
    image = ImageOps.autocontrast(image)
    image = image.filter(ImageFilter.SHARPEN)
    image = image.filter(ImageFilter.MedianFilter(size=3))
    return image.point(lambda px: 255 if px > 165 else 0)

def save_temp_image(image: Image.Image) -> Path:
    temp_file = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    temp_path = Path(temp_file.name)
    temp_file.close()
    image.save(temp_path)
    return temp_path

def convert_heic_to_png(image_path: Path) -> Path:
    temp_file = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    temp_path = Path(temp_file.name)
    temp_file.close()
    result = subprocess.run(
        ['sips', '-s', 'format', 'png', str(image_path), '--out', str(temp_path)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        temp_path.unlink(missing_ok=True)
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or 'sips failed')
    return temp_path

def prepare_source_image(image_path: Path):
    if image_path.suffix.lower() in {'.heic', '.heif'}:
        return convert_heic_to_png(image_path), True
    return image_path, False

def build_variants(image_path: Path):
    source = ImageOps.exif_transpose(Image.open(image_path))
    width, height = source.size
    center_crop = source.crop((int(width * 0.28), int(height * 0.18), int(width * 0.82), int(height * 0.72)))
    tight_crop = source.crop((int(width * 0.34), int(height * 0.24), int(width * 0.72), int(height * 0.58)))

    variants = []
    for base_image in (source, center_crop, tight_crop):
        for angle in (0, 90, 180, 270):
            rotated = base_image.rotate(angle, expand=True)
            variants.append(save_temp_image(preprocess_image(rotated)))
    return variants

def run_tesseract(image_path: Path, psm: int, digits_only: bool = False) -> str:
    command = ['tesseract', str(image_path), 'stdout', '--psm', str(psm), '-l', 'eng']
    if digits_only:
        command.extend(['-c', 'tessedit_char_whitelist=0123456789UserID:userid'])
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or 'tesseract failed')
    return result.stdout

def normalize_text(text: str) -> str:
    return (
        text.replace('|', 'I')
        .replace('—', '-')
        .replace('–', '-')
        .replace('User 1D', 'User ID')
        .replace('Userid', 'User ID')
        .replace('user \\D', 'user ID')
    )

def score_candidate(candidate: str, context: str) -> int:
    score = 0
    if re.search(r'user\s*id', context, re.IGNORECASE):
        score += 8
    if re.search(r'moomoo', context, re.IGNORECASE):
        score += 5
    if re.search(r'\bid\b', context, re.IGNORECASE):
        score += 3
    if re.fullmatch(r'\d{8}', candidate):
        score += 10
    return score

def extract_candidates(text: str):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    candidates = []
    for index, line in enumerate(lines):
        nearby = '\n'.join(lines[max(0, index - 1): min(len(lines), index + 2)])
        line_matches_keyword = bool(KEYWORD_PATTERN.search(nearby))
        for pattern in ID_PATTERNS:
            for match in pattern.finditer(line):
                value = match.group(1)
                score = score_candidate(value, nearby)
                if line_matches_keyword:
                    score += 3
                candidates.append({'value': value, 'score': score, 'line': line})
    deduped = {}
    for candidate in candidates:
        current = deduped.get(candidate['value'])
        if current is None or candidate['score'] > current['score']:
            deduped[candidate['value']] = candidate
    return sorted(deduped.values(), key=lambda item: item['score'], reverse=True)

def choose_best_candidate(text_variants):
    for text in text_variants:
        for pattern in DIRECT_PATTERNS:
            match = pattern.search(text)
            if match:
                return match.group(1), extract_candidates(text)
    all_candidates = []
    for text in text_variants:
        all_candidates.extend(extract_candidates(text))
    if not all_candidates:
        return None, []
    best = sorted(all_candidates, key=lambda item: item['score'], reverse=True)[0]['value']
    ranked = []
    seen = set()
    for item in sorted(all_candidates, key=lambda item: item['score'], reverse=True):
        if item['value'] not in seen:
            ranked.append(item)
            seen.add(item['value'])
    return best, ranked

def extract_from_image(image_path: Path):
    source_image_path, should_cleanup_source = prepare_source_image(image_path)
    variants = build_variants(source_image_path)
    texts = []
    try:
        texts.append(normalize_text(run_tesseract(source_image_path, psm=6)))
        for variant_path in variants:
            texts.append(normalize_text(run_tesseract(variant_path, psm=6)))
            texts.append(normalize_text(run_tesseract(variant_path, psm=11)))
            texts.append(normalize_text(run_tesseract(variant_path, psm=6, digits_only=True)))
    finally:
        if should_cleanup_source:
            source_image_path.unlink(missing_ok=True)
        for variant_path in variants:
            variant_path.unlink(missing_ok=True)

    best_candidate, ranked = choose_best_candidate(texts)
    return {
        'file_name': image_path.name,
        'user_id': best_candidate or '',
        'success_photo_name': image_path.name if best_candidate else '',
        'failed_photo_name': '' if best_candidate else image_path.name,
        'status': 'success' if best_candidate else 'failed',
        'top_candidates': ','.join(candidate['value'] for candidate in ranked[:3]),
    }


## 1. 扫描前检查文件夹

先运行下面这个 cell，确认照片数量和前几个文件名无误。

In [ ]:
image_files = list_image_files(INPUT_DIR)
print(f'Folder: {INPUT_DIR}')
print(f'Total images: {len(image_files)}')
print('Head:')
for path in image_files[:20]:
    print('-', path.name)

## 2. 提取 User ID 并生成 CSV

确认无误后再运行下面这个 cell。

In [ ]:
rows = [extract_from_image(image_path) for image_path in image_files]

with OUTPUT_CSV.open('w', newline='', encoding='utf-8-sig') as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=['file_name', 'user_id', 'success_photo_name', 'failed_photo_name', 'status', 'top_candidates'],
    )
    writer.writeheader()
    writer.writerows(rows)

success_count = sum(1 for row in rows if row['status'] == 'success')
failed_count = len(rows) - success_count
print(f'Processed {len(rows)} images')
print(f'Success: {success_count}')
print(f'Failed: {failed_count}')
print(f'CSV saved to: {OUTPUT_CSV}')

In [ ]:
rows[:10]